In [ ]:
# Empty code cell required for "ruff" extension

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import data_path
from src.spark_utils import get_spark
import pyspark.sql.functions as F
from src.datasets import read_qqp, read_qqp_benchmark, read_feature_benchmark
from src.metrics import ModelMetrics, extract_metrics

%load_ext autoreload
%autoreload 2

spark = get_spark()

In [ ]:
from src.datasets import read_c5
from src.fingerprint import create_fingerprints

df = read_c5()
df = df.withColumn("length", F.length("data"))
df.show()

In [ ]:
from src.datasets import read_c5
from src.fingerprint import create_fingerprints

df = read_c5()
fp_df = create_fingerprints(df=df, model="embeddings", input_col="data", output_col="fingerprint", fingerprint_size=128, other=None)
fp_df.show()

In [ ]:
from src.near_duplicates import detect_near_duplicates_benchmark
from src.fingerprint import create_fingerprints


df = read_qqp().limit(1000)
df = create_fingerprints(df, "embeddings", "data1", "fingerprint1", 64, None)
df = create_fingerprints(df, "embeddings", "data2", "fingerprint2", 64, None)
df.show()
# metrics = detect_near_duplicates_benchmark(df, "fingerprint1", "fingerprint2", "is_near_duplicate", 5)

In [ ]:
from src.near_duplicates import partitions_schema


f = 64
p_min = 10
K = 6
d = 20

partitions_schema(f=f, K=K, d=d, p_min=p_min)

In [ ]:
from pyspark.sql.types import StructField, StructType, BinaryType, IntegerType

from src.near_duplicates import detect_near_duplicate_v2

schema = StructType([
    StructField("ID", IntegerType(), nullable=False),
    StructField("fingerprint", BinaryType(), nullable=False)
])

data1 = [
    (1, bytes.fromhex("0123456789ABCDEF")),
    (2, bytes.fromhex("00000000FFFFFFFF")),
    (3, bytes.fromhex("FFFFFFFF00000000"))
]

# Dati per il secondo DataFrame
data2 = [
    (10, bytes.fromhex("00000111FFFFFFFF")),
    (20, bytes.fromhex("8877000044332211")),
    (30, bytes.fromhex("CAFEBABE00112233"))
]

df1 = spark.createDataFrame(data=data1, schema=schema)
df2 = spark.createDataFrame(data=data2, schema=schema)

detect_near_duplicate_v2(df1, df2, 64, 3, "ID", "fingerprint", p_min=10, corpus_size=3).show()

In [ ]:
from src.datasets import read_llm_paraphrases, read_wikipedia, read_c5
from src.near_duplicates import detect_near_duplicates_pairs
df = read_qqp()
df1 = df.filter(F.col("is_near_duplicate") == F.lit(True)).limit(3)
df2 = df.filter(F.col("is_near_duplicate") == F.lit(False)).limit(3)
df = df1.union(df2)
df.toPandas().to_csv("qqp_sample.csv")

In [ ]:
from src.datasets import read_wikipedia


df = read_wikipedia(variant="10percent")
print(df.rdd.getNumPartitions()) 

df_limited = df.limit(300)
print(df_limited.rdd.getNumPartitions()) 

In [ ]:
spark.stop()